# Pan-African Soybean G×E Challenge — Submission Notebook

**Competition:** DataTour 2026 · dataafriquehub.org  
**Task:** predict soybean grain yield (kg/ha) in environments never seen in training  
**Metric:** RMSE

---

## How this model was chosen

Every submission, and what it taught:

| # | model | unseen-loc | leaderboard |
|---|---|---|---|
| 111 | LightGBM + XGBoost | — | 1121.92 |
| 112 | CatBoost 90% | — | 1080.25 |
| 113 | CatBoost blend, α=1.15 | ~1005.9 | **1069.83** |
| 114 | LightGBM 80%, α=1.00 | 1030.2 | 1100.02 |
| this | CatBoost top-3 configs, α=1.25 | **998.0** | — |

Two conclusions come straight from that table: **more CatBoost is better**, and
**α > 1 helps**. Submission 114 tested the opposite of both and lost 30 points.

### The validation metric

Plain out-of-fold RMSE is misleading here — a validation environment can reuse a
training *location* under a different year, which no test environment can do.
Scoring only the rows whose location is absent from their own training fold
ranks submissions 113 and 114 correctly and in roughly the right proportion
(1005.9 → 1069.83, 1030.2 → 1100.02), so that subset is the objective throughout
this notebook.

### Why α > 1

The model is badly **under-dispersed**, measured on out-of-fold predictions:

| component | predicted std | actual std |
|---|---|---|
| between-environment | 478 | 917 |
| within-environment | 122 | 603 |

Because climate and soil predict the environment mean at only r = 0.47, the model
hedges toward the global mean and the spread of predictions collapses. Rescaling
away from the mean by α = 1.25 restores it and is worth ~10 kg/ha.

Fitting *separate* factors for the two components was tried and did **not** help:
the fit returned a_between = a_within = 1.20, so a single uniform α was already
optimal.

### Measured and rejected

| idea | outcome |
|---|---|
| XGBoost in the blend | 1009 OOF, dragged the ensemble down |
| LightGBM as the primary model | 1100.02 on the board, the worst recent result |
| log-transform of the target | skew 0.50 → −1.02, strictly worse |
| two-stage environment-mean + deviation | 1025 vs 956 |
| variety-panel fingerprint features | 960.8 vs 959.6 |
| Finlay-Wilkinson stability features | no gain |
| 105-feature set | 1009.9 vs 1007.3 for 102 |
| dropping the location features | 1013.4 vs 1007.3 — they earn their place |
| folds grouped by `loc` | makes `loc_mean_yield` 100% missing while 47% of test rows sit at a known location |
| per-component α | a_between = a_within, no gain over uniform |
| label-encoding `variety_id` to a float | **bug** — 365 varieties read as an ordered number |
| row-order / `id` leakage | none, rank-correlation ≈ 0.05 |

Fully seeded — re-running reproduces `submission.csv` exactly.

## 1. Setup

In [ ]:
import glob, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from catboost import CatBoostRegressor, Pool
warnings.filterwarnings('ignore')

N_FOLDS, K_NBR = 5, 5
SEEDS   = [42, 202, 777]
CONFIGS = [(8, 3), (8, 6), (9, 3)]   # top 3 of a 3-seed sweep, equal weight
ALPHA   = 1.25                       # dispersion correction, fitted on unseen-loc
np.random.seed(42)

_c = glob.glob('/kaggle/input/**/train.csv', recursive=True)
if _c:
    TRAIN_PATH  = _c[0]
    TEST_PATH   = TRAIN_PATH.replace('train.csv', 'test.csv')
    SAMPLE_PATH = TRAIN_PATH.replace('train.csv', 'sample_submission.csv')
else:
    TRAIN_PATH, TEST_PATH, SAMPLE_PATH = 'train.csv', 'test.csv', 'sample_submission.csv'

TARGET, ID, ENV, VAR = 'yield_kg_ha', 'id', 'environment_id', 'variety_id'
rmse = lambda a, b: float(np.sqrt(np.mean((a - b) ** 2)))
print('train:', TRAIN_PATH)

## 2. Load data

In [ ]:
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
gmean = train[TARGET].mean()

print(f'train {train.shape} | test {test.shape}')
print(f'environments  train {train[ENV].nunique()} | test {test[ENV].nunique()} | '
      f'overlap {len(set(train[ENV]) & set(test[ENV]))}')

em = train.groupby(ENV)[TARGET].mean()
print(f'global mean {gmean:.0f} | target std {train[TARGET].std():.0f}')
print(f'between-env std {em.std():.0f} | '
      f'within-env std {(train[TARGET] - train[ENV].map(em)).std():.0f}')
print(f'predicting the global mean everywhere would score {train[TARGET].std():.0f}')
print(f'test rows at a location seen in train: '
      f'{test["loc"].isin(set(train["loc"])).mean()*100:.0f}%')

## 3. Static features

`is_irrigated` keys on `"irrig"`. `RAINFED` takes three values — Rainfed,
Irrigation, Supplementary — so defining the flag as "anything not rainfed"
wrongly folds Supplementary in with full irrigation (1501 rows instead of 914).

In [ ]:
def add_static(df):
    df = df.copy()
    df['SOWING_dt']   = pd.to_datetime(df['SOWING'], dayfirst=True, errors='coerce')
    df['sow_month']   = df['SOWING_dt'].dt.month
    df['sow_doy']     = df['SOWING_dt'].dt.dayofyear
    df['sow_doy_sin'] = np.sin(2 * np.pi * df['sow_doy'] / 365.25)
    df['sow_doy_cos'] = np.cos(2 * np.pi * df['sow_doy'] / 365.25)
    df['hemisphere']  = np.where(df['LAT'] >= 0, 'N', 'S')
    df['abs_lat']     = df['LAT'].abs()
    df['is_irrigated'] = (df['RAINFED'].astype(str).str.lower()
                            .str.contains('irrig')).astype(int)
    df['temp_annual_mean']   = df['wc2.1_30s_bio_1']
    df['precip_annual']      = df['wc2.1_30s_bio_12']
    df['precip_seasonality'] = df['wc2.1_30s_bio_15']
    df['temp_range']         = df['wc2.1_30s_bio_5'] - df['wc2.1_30s_bio_6']
    df['aridity_proxy']      = df['precip_annual'] / (df['temp_annual_mean'] + 10)
    for var in ['clay', 'sand', 'silt', 'nitrogen', 'phh2o', 'soc']:
        cols = [c for c in df.columns if c.startswith(var + '_')]
        if cols:
            df[f'{var}_mean_depth'] = df[cols].mean(axis=1)
    return df

train, test = add_static(train), add_static(test)
FOLDS = list(GroupKFold(n_splits=N_FOLDS).split(train, groups=train[ENV]))
print('static features done')

## 4. Leak-free target encodings

Variety and location statistics come from out-of-fold data on train (a fold never
sees its own environments) and from the full training set for test, since no test
environment appears in train.

In [ ]:
VAGG = dict(variety_mean_yield='mean', variety_median_yield='median',
            variety_std_yield='std',  variety_n_trials='count')

for c in VAGG:
    train[c] = np.nan
for fit, val in FOLDS:
    st = train.iloc[fit].groupby(VAR)[TARGET].agg(**VAGG)
    for c in st.columns:
        train.loc[train.index[val], c] = train.iloc[val][VAR].map(st[c]).values

vmed = train['variety_std_yield'].median()
train['variety_mean_yield']   = train['variety_mean_yield'].fillna(gmean)
train['variety_median_yield'] = train['variety_median_yield'].fillna(gmean)
train['variety_std_yield']    = train['variety_std_yield'].fillna(vmed)
train['variety_n_trials']     = train['variety_n_trials'].fillna(1)

test = test.merge(train.groupby(VAR)[TARGET].agg(**VAGG), on=VAR, how='left')
test['variety_mean_yield']   = test['variety_mean_yield'].fillna(gmean)
test['variety_median_yield'] = test['variety_median_yield'].fillna(gmean)
test['variety_std_yield']    = test['variety_std_yield'].fillna(vmed)
test['variety_n_trials']     = test['variety_n_trials'].fillna(0)

train['loc_mean_yield'] = np.nan
train['loc_n_trials']   = np.nan
for fit, val in FOLDS:
    st = train.iloc[fit].groupby('loc')[TARGET].agg(loc_mean_yield='mean',
                                                    loc_n_trials='count')
    for c in st.columns:
        train.loc[train.index[val], c] = train.iloc[val]['loc'].map(st[c]).values
train['loc_mean_yield'] = train['loc_mean_yield'].fillna(gmean)
train['loc_n_trials']   = train['loc_n_trials'].fillna(0)

lfull = train.groupby('loc').agg(loc_mean_yield=(TARGET, 'mean'),
                                 loc_n_trials=(TARGET, 'count'),
                                 loc_lat=('LAT', 'first'), loc_lon=('LON', 'first'))
test = test.merge(lfull[['loc_mean_yield', 'loc_n_trials']], on='loc', how='left')
test['loc_mean_yield'] = test['loc_mean_yield'].fillna(gmean)
test['loc_n_trials']   = test['loc_n_trials'].fillna(0)
print('target encodings done')

## 5. Spatial k-NN and G×E interactions

In [ ]:
def haversine(a1, o1, a2, o2):
    a1, o1, a2, o2 = map(np.radians, [a1, o1, a2, o2])
    h = np.sin((a2 - a1) / 2) ** 2 + np.cos(a1) * np.cos(a2) * np.sin((o2 - o1) / 2) ** 2
    return 2 * 6371 * np.arcsin(np.sqrt(np.clip(h, 0, 1)))

known = lfull.reset_index()

def knn_yield(lat, lon, exclude=None):
    d = haversine(lat, lon, known['loc_lat'].values, known['loc_lon'].values)
    if exclude is not None:
        d = np.where(known['loc'].values == exclude, np.inf, d)
    d = np.where(d == 0, 1e-3, d)
    i = np.argsort(d)[:K_NBR]
    return np.average(known['loc_mean_yield'].values[i], weights=1 / d[i])

# a training row excludes its own site, mirroring an unseen location at test time
train['spatial_knn_yield'] = [knn_yield(r.LAT, r.LON, r.loc) for r in train.itertuples()]
test['spatial_knn_yield']  = [knn_yield(r.LAT, r.LON)        for r in test.itertuples()]

for df in (train, test):
    df['gxe_variety_temp']    = df['variety_mean_yield'] * df['temp_annual_mean']
    df['gxe_variety_precip']  = df['variety_mean_yield'] * df['precip_annual']
    df['gxe_variety_aridity'] = df['variety_mean_yield'] * df['aridity_proxy']
print('spatial + G×E features done')

## 6. Categoricals stay categorical

Label-encoding `variety_id` to a float was the costliest bug of this competition:
the model then reads 365 varieties as an *ordered number* and splits on
meaningless thresholds. CatBoost receives them as strings and handles them natively.

In [ ]:
CATS = [c for c in ['COUNTRY','SEASON','RAINFED','SOURCE','COMPANY','hemisphere', VAR]
        if c in train.columns]
DROP  = [ID, TARGET, ENV, 'SOWING', 'SOWING_dt', 'loc']
feats = [c for c in train.columns if c not in DROP]

X, Xt = train[feats].copy(), test[feats].copy()
for c in CATS:
    X[c], Xt[c] = X[c].astype(str), Xt[c].astype(str)

cat_idx = [feats.index(c) for c in CATS]
y = train[TARGET].values
print(f'{len(feats)} features | {len(CATS)} categorical -> {CATS}')

## 7. The honest validation subset

Rows whose location never appears in their own training fold. This is the metric
every choice below was made against.

In [ ]:
newloc = np.zeros(len(y), dtype=bool)
for fit, val in FOLDS:
    newloc[val] = ~train.iloc[val]['loc'].isin(set(train.iloc[fit]['loc'])).values
print(f'unseen-location rows: {newloc.sum()} / {len(y)} ({100*newloc.mean():.0f}%)')

## 8. CatBoost — three configs, three seeds each

The depth/l2 grid was first scored on a single seed, but seed 42 turned out to be
lucky (953.8 against 959.3 and 958.3 on its siblings), so every config was re-run
over three seeds. Depth 8 / l2 3 survived that check as the genuine winner; the
next two are averaged in to blunt selection noise.

In [ ]:
oof_total, te_total = np.zeros(len(y)), np.zeros(len(Xt))

for depth, l2 in CONFIGS:
    oof_c, te_c = np.zeros(len(y)), np.zeros(len(Xt))
    for s in SEEDS:
        o = np.zeros(len(y))
        for fit, val in FOLDS:
            m = CatBoostRegressor(
                iterations=4000, learning_rate=0.02, depth=depth, l2_leaf_reg=l2,
                loss_function='RMSE', eval_metric='RMSE', random_seed=s,
                verbose=0, early_stopping_rounds=200,
                cat_features=cat_idx, one_hot_max_size=8)
            m.fit(Pool(X.iloc[fit], y[fit], cat_features=cat_idx),
                  eval_set=Pool(X.iloc[val], y[val], cat_features=cat_idx),
                  use_best_model=True)
            o[val] = m.predict(X.iloc[val])
            te_c  += m.predict(Xt) / (len(FOLDS) * len(SEEDS))
        oof_c += o / len(SEEDS)
    print(f'  depth={depth} l2={l2}:  all {rmse(y, oof_c):.1f} | '
          f'unseen-loc {rmse(y[newloc], oof_c[newloc]):.1f}')
    oof_total += oof_c / len(CONFIGS)
    te_total  += te_c  / len(CONFIGS)

print(f'\nensemble:  all {rmse(y, oof_total):.1f} | '
      f'unseen-loc {rmse(y[newloc], oof_total[newloc]):.1f}')

## 9. Dispersion correction

The predictions are too tightly clustered around the mean, so rescaling away from
it reduces error. α is fitted on the unseen-location subset.

In [ ]:
env_tr   = train[ENV].values
pred_env = pd.Series(oof_total).groupby(env_tr).transform('mean').values
true_env = pd.Series(y).groupby(env_tr).transform('mean').values
print('dispersion of out-of-fold predictions vs reality:')
print(f'  between-env std  predicted {pd.Series(pred_env).groupby(env_tr).first().std():.0f}'
      f'  actual {pd.Series(true_env).groupby(env_tr).first().std():.0f}')
print(f'  within-env  std  predicted {(oof_total - pred_env).std():.0f}'
      f'  actual {(y - true_env).std():.0f}')

print('\nalpha sweep on the unseen-location subset:')
for a in [1.00, 1.10, 1.15, 1.20, 1.25, 1.30, 1.35]:
    sc = gmean + a * (oof_total - gmean)
    mark = '  <- used' if abs(a - ALPHA) < 1e-9 else ''
    print(f'  alpha={a:.2f}  unseen-loc {rmse(y[newloc], sc[newloc]):.1f}   '
          f'all {rmse(y, sc):.1f}{mark}')

## 10. Write the submission

In [ ]:
final = np.clip(gmean + ALPHA * (te_total - gmean), 0, None)

sample = pd.read_csv(SAMPLE_PATH)
tcol   = [c for c in sample.columns if c != ID][0]
sub    = sample[[ID]].copy()
sub[tcol] = sub[ID].map(dict(zip(test[ID], final)))

assert sub[tcol].isna().sum() == 0, 'missing predictions'
assert len(sub) == len(sample),     'row count differs from sample_submission'
assert list(sub.columns) == list(sample.columns), 'column layout differs'

sub.to_csv('submission.csv', index=False)
print(f'submission.csv written — {len(sub)} rows')
print(f'preds  min {final.min():.0f} | mean {final.mean():.0f} | '
      f'max {final.max():.0f} | std {final.std():.0f}')
print(sub.head().to_string(index=False))

---

## Where the remaining error lives

Predicting the global mean for every row scores 1083. This model reaches ~998 on
the unseen-location subset. That narrow margin is the whole story: the
environment mean is 83% of the variance in yield and the supplied climate and
soil layers predict it at only r = 0.47.

The within-environment picture is equally constrained. Variety explains about 12%
of the variation inside an environment (corr 0.35), so most of that 603 kg/ha of
spread is irreducible noise, not missed signal. No amount of tuning reaches it.

Checked and ruled out as explanations for the top of the leaderboard: row-order
leakage, `id`-hash leakage, and shared `(loc, year)` keys between train and test —
all clean.

What would actually move the score, in order of expected value:

1. **In-season weather for the actual trial year** instead of 30-year WorldClim
   normals. Two trials at one site in different years currently receive *identical*
   climate features despite different yields, so the model cannot separate a good
   season from a bad one. This is the single largest gap.
2. **Management intensity** — fertiliser, spacing, plot size — which separates
   high- and low-yielding trials at the same site.
3. **Maturity group / variety pedigree**, so `variety_id` generalises to varieties
   appearing in only one or two trials.
4. **Days from sowing to the local season start**, rather than raw day of year.